# Stage 1 — Run classifiers on the Ethiopia moderation dataset

Produces predictions from three classifiers on the **838 audit-eligible rows** of the cleaned **951-row** corpus (Amharic + Afaan Oromo), saved to `predictions.xlsx` (keyed by `PostID`) for Stage 2.

1. **Perspective API** — generic baseline; reads the **English translation** (Perspective lacks Amharic/Afaan Oromo). API key required.
2. **AfriHate (AfroXLMR)** — fine-tuned here on the AfriHate dataset (no ready-made checkpoint exists); reads native **OriginalText**.
3. **Amharic mBERT** — Amharic-only; reads native **OriginalText**, Amharic rows only.

**Input:** upload `corpus_SCORING_PRIVATE_unmasked.xlsx` (UNMASKED — never publish it).
**Setup:** Runtime → Change runtime type → **T4 GPU**.


## 1. Install dependencies


In [10]:
!pip -q install transformers datasets torch requests openpyxl pandas scikit-learn --upgrade


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 836.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [41]:
!pip -q install transformers datasets requests openpyxl scikit-learn "pandas==2.2.2"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.


In [1]:
!pip install -q -U transformers accelerate datasets

## 2. Upload the unmasked scoring file


In [5]:
import pandas as pd
CORPUS_PATH = 'corpus_SCORING_PRIVATE_unmasked[1].xlsx'

In [22]:
import pandas as pd
CORPUS_PATH = 'corpus_SCORING_PRIVATE_unmasked.xlsx'
print('Using file:', CORPUS_PATH)

Using file: corpus_SCORING_PRIVATE_unmasked.xlsx


In [3]:
import os
print(os.listdir('/content'))

['.config', 'sample_data']


In [13]:
import pandas as pd
df = pd.read_excel('perspective_results.xlsx')
print('df:', len(df), '| persp scored:', df['persp_score'].notna().sum())

df: 838 | persp scored: 687


## 3. Load corpus and select audit-eligible rows


In [6]:
import pandas as pd
df = pd.read_excel('perspective_results.xlsx')
print('Reloaded:', len(df), '| persp scored:', df['persp_score'].notna().sum())

Reloaded: 838 | persp scored: 687


In [44]:
CORPUS_PATH = 'corpus_SCORING_PRIVATE_unmasked[1].xlsx'
TEXT_NATIVE = 'OriginalText'        # AfriHate + Amharic mBERT read this
TEXT_PERSP  = 'EnglishTranslation'  # Perspective reads this
GOLD='Label3Class'; LANG='Language'; AUDIT='IncludeInAudit'; POS_GOLD='Hate'

df = pd.read_excel(CORPUS_PATH, sheet_name='Dataset')
df = df[df[AUDIT]==True].copy()
df = df[df[TEXT_NATIVE].notna()].reset_index(drop=True)
df['gold_bin'] = (df[GOLD].astype(str).str.strip()==POS_GOLD).astype(int)
print(f'Audit-eligible rows: {len(df)} (Hate={df.gold_bin.sum()}, not-Hate={(df.gold_bin==0).sum()})')
print('Languages:', dict(df[LANG].value_counts()))


Audit-eligible rows: 838 (Hate=597, not-Hate=241)
Languages: {'Amharic': np.int64(470), 'Afan Oromo': np.int64(368)}


In [11]:
import pandas as pd
df = pd.read_excel('perspective_results.xlsx')
print('df:', len(df), '| persp scored:', df['persp_score'].notna().sum())

FileNotFoundError: [Errno 2] No such file or directory: 'perspective_results.xlsx'

## 4. Classifier 1 — Perspective API (English translation)
Set your key when prompted. Unscored rows are kept as `None` (coverage gap).


In [45]:
import requests, time

PERSPECTIVE_API_KEY = "YOUR_PERSPECTIVE_API_KEY"   # <-- paste your key between the quotes

THRESH = 0.5
URL = 'https://commentanalyzer.googleapis.com/v1alpha1/comments:analyze?key=' + PERSPECTIVE_API_KEY

# --- quick test of ONE call first ---
test = requests.post(URL, json={'comment': {'text': 'I hate you'},
                                'requestedAttributes': {'TOXICITY': {}}}, timeout=30)
print('Test status:', test.status_code)
if test.status_code != 200:
    print('KEY PROBLEM — response:', test.json())
else:
    print('Key works! Running all rows...')
    def persp(t):
        try:
            r = requests.post(URL, json={'comment': {'text': str(t)[:3000]},
                                         'requestedAttributes': {'TOXICITY': {}}}, timeout=30)
            return r.json()['attributeScores']['TOXICITY']['summaryScore']['value'] if r.status_code==200 else None
        except Exception:
            return None
    scores = []
    for i, t in enumerate(df[TEXT_PERSP].tolist()):
        scores.append(persp(t)); time.sleep(1.1)
        if (i+1) % 50 == 0: print(f'  Perspective {i+1}/838')
    df['persp_score'] = scores
    df['persp_pred'] = [None if s is None else int(s>=THRESH) for s in scores]
    print('Done. Perspective scored', df['persp_pred'].notna().sum(), 'of 838')

Test status: 200
Key works! Running all rows...
  Perspective 50/838
  Perspective 100/838
  Perspective 150/838
  Perspective 200/838
  Perspective 250/838
  Perspective 300/838
  Perspective 350/838
  Perspective 400/838
  Perspective 450/838
  Perspective 500/838
  Perspective 550/838
  Perspective 600/838
  Perspective 650/838
  Perspective 700/838
  Perspective 750/838
  Perspective 800/838
Done. Perspective scored 687 of 838


In [46]:
df.to_excel('perspective_results.xlsx', index=False)
print('Saved. persp_score rows:', df['persp_score'].notna().sum())
from google.colab import files
files.download('perspective_results.xlsx')

Saved. persp_score rows: 687


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5a. Classifier 2 — fine-tune AfroXLMR on AfriHate
There is no ready-made AfriHate classifier, so we fine-tune AfroXLMR on the AfriHate dataset (`afrihate/afrihate`), following Muhammad et al. (2025). We train on the **Amharic** split (AfriHate's Oromo has only a *test* set, so Afaan Oromo predictions rely on cross-lingual transfer — note this in your results). Labels: {hate, abusive, neutral}.

`afro-xlmr-base` is the reliable default on a free T4; switch `BASE_MODEL` to `Davlan/afro-xlmr-large-76L` to match the paper's best (heavier; reduce batch size if you hit out-of-memory).


In [14]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import Dataset
import pandas as pd, numpy as np, torch
from sklearn.metrics import f1_score

BASE_MODEL = 'Davlan/afro-xlmr-base'   # paper-best alt: 'Davlan/afro-xlmr-large-76L'
EPOCHS = 3; MAXLEN = 128; BATCH = 16

# --- load local AfriHate tsv files ---
tr = pd.read_csv('train.tsv', sep='\t')
dv = pd.read_csv('dev.tsv',   sep='\t')
print('train columns:', list(tr.columns))
print('train shape:', tr.shape)
print(tr.head(3))

# --- auto-detect text + label columns ---
TEXTC = next((c for c in ['tweet','text','content'] if c in tr.columns), tr.columns[0])
LABC  = next((c for c in ['label','labels','class'] if c in tr.columns), tr.columns[-1])
print('text col:', TEXTC, '| label col:', LABC)
print('label values:', sorted(tr[LABC].astype(str).unique()))
# --- encode labels (string -> int), keep names for inference ---
LABEL_NAMES = sorted(tr[LABC].astype(str).unique())          # ['Abuse','Hate','Normal']
lab2id = {n:i for i,n in enumerate(LABEL_NAMES)}
print('label mapping:', lab2id)

def to_ds(frame):
    f = frame[[TEXTC, LABC]].dropna().copy()
    f['labels'] = f[LABC].astype(str).map(lab2id)
    return Dataset.from_pandas(f[[TEXTC,'labels']], preserve_index=False)

train_ds_raw, val_ds_raw = to_ds(tr), to_ds(dv)

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
def tok_fn(b): return tok(b[TEXTC], truncation=True, max_length=MAXLEN)
train_ds = train_ds_raw.map(tok_fn, batched=True)
val_ds   = val_ds_raw.map(tok_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=len(LABEL_NAMES))
model.config.id2label = {i:n for n,i in lab2id.items()}
model.config.label2id = lab2id

def compute_metrics(p):
    pr = np.argmax(p.predictions, axis=1)
    return {'macro_f1': f1_score(p.label_ids, pr, average='macro')}

args = TrainingArguments(output_dir='afrihate_model', num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=32, learning_rate=2e-5,
    fp16=torch.cuda.is_available(), eval_strategy='epoch', save_strategy='no',
    logging_steps=50, report_to='none')
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tok, data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
trainer.train()
print('Validation:', trainer.evaluate())

train columns: ['id', 'tweet', 'label']
train shape: (3467, 3)
                    id                                              tweet  \
0  train_amharic_00001  @USER @USER @USER አላማህን አውቄብሃለው ማለት ነው ??? እኔ ...   
1  train_amharic_00002  @USER ውጤታማ ከሆነ ዳግም ማፈናቀል ጭካኔአዊ ግድያ ከቆመ አንድ ሰው ...   
2  train_amharic_00003  @USER ሁለት ቀን ሆነኝ ከተኛዉ! እዉነት የሰራዊት ንጉስ ይፍረድ ???...   

    label  
0   Abuse  
1  Normal  
2    Hate  
text col: tweet | label col: label
label values: ['Abuse', 'Hate', 'Normal']
label mapping: {'Abuse': 0, 'Hate': 1, 'Normal': 2}


Map:   0%|          | 0/3467 [00:00<?, ? examples/s]

Map:   0%|          | 0/744 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,0.754894,0.714680,0.684745
2,0.608733,0.691528,0.710506
3,0.496457,0.709045,0.714197


Training Loss,Validation Loss,Epoch,Macro F1
0.496457,0.709045,3,0.714197


Validation: {'eval_loss': 0.7090449333190918, 'eval_macro_f1': 0.7141974498158472}


In [5]:
trainer.save_model('afrihate_model')
tok.save_pretrained('afrihate_model')
print('Model saved to afrihate_model/')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to afrihate_model/


In [15]:
trainer.save_model('afrihate_model')
tok.save_pretrained('afrihate_model')
# zip it and download to your computer
import shutil
shutil.make_archive('afrihate_model', 'zip', 'afrihate_model')
from google.colab import files
files.download('afrihate_model.zip')
print('Model saved AND downloading as afrihate_model.zip')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Model saved AND downloading as afrihate_model.zip


In [16]:
!pip install -q -U transformers accelerate datasets

In [22]:
import shutil, os
# clear any half-downloaded afrihate cache
cache = os.path.expanduser('~/.cache/huggingface/datasets')
for d in os.listdir(cache) if os.path.exists(cache) else []:
    if 'afrihate' in d.lower():
        shutil.rmtree(os.path.join(cache, d), ignore_errors=True)
        print('cleared:', d)
print('done')

cleared: afrihate___afrihate
done


In [33]:
raw = load_dataset('afrihate/afrihate', TRAIN_LANGS[0])

FileNotFoundError: An error happened while trying to locate the file on the Hub and we cannot find the requested files in the local cache. Please check your connection and try again or make sure your Internet connection is on.

In [34]:
raw = load_dataset('afrihate/afrihate', TRAIN_LANGS[0], download_mode='force_redownload')

ValueError: Force download failed due to the above error.

In [35]:
from huggingface_hub import whoami
from datasets import get_dataset_config_names

# 1) confirm WHO you're logged in as
print("Logged in as:", whoami()["name"])

# 2) test access to the dataset directly
try:
    cfgs = get_dataset_config_names("afrihate/afrihate")
    print("ACCESS OK — configs:", cfgs)
except Exception as e:
    print("ACCESS FAILED:", type(e).__name__, str(e)[:200])

Logged in as: endalkchala
ACCESS OK — configs: ['amh', 'arq', 'ary', 'hau', 'ibo', 'kin', 'orm', 'som', 'swa', 'pcm', 'tir', 'twi', 'xho', 'yor', 'zul']


In [36]:
import shutil, os
cache = os.path.expanduser('~/.cache/huggingface/datasets')
if os.path.exists(cache):
    for d in os.listdir(cache):
        if 'afrihate' in d.lower():
            shutil.rmtree(os.path.join(cache, d), ignore_errors=True)
            print('cleared:', d)
print('done')

cleared: afrihate___afrihate
done


In [26]:
raw = load_dataset('afrihate/afrihate', TRAIN_LANGS[0])

FileNotFoundError: An error happened while trying to locate the file on the Hub and we cannot find the requested files in the local cache. Please check your connection and try again or make sure your Internet connection is on.

In [31]:
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")

In [17]:
from huggingface_hub import login
login()

In [37]:
import shutil, os
cache = os.path.expanduser('~/.cache/huggingface/datasets')
if os.path.exists(cache):
    for d in os.listdir(cache):
        if 'afrihate' in d.lower():
            shutil.rmtree(os.path.join(cache, d), ignore_errors=True)
            print('cleared:', d)
print('cache clean')

cache clean


In [38]:
print('df rows:', len(df), '| persp scored:', df['persp_score'].notna().sum())

df rows: 838 | persp scored: 687


In [40]:
raw = load_dataset('afrihate/afrihate', TRAIN_LANGS[0])

FileNotFoundError: An error happened while trying to locate the file on the Hub and we cannot find the requested files in the local cache. Please check your connection and try again or make sure your Internet connection is on.

In [42]:
from huggingface_hub import logout, login
logout()
login(token="YOUR_HF_TOKEN")

In [43]:
from huggingface_hub import hf_hub_download
try:
    p = hf_hub_download("afrihate/afrihate", "data/amh/train.tsv", repo_type="dataset")
    print("SUCCESS — token can read gated file:", p)
except Exception as e:
    print("STILL BLOCKED:", str(e)[:250])

STILL BLOCKED: An error happened while trying to locate the file on the Hub and we cannot find the requested files in the local cache. Please check your connection and try again or make sure your Internet connection is on.


In [1]:
import os
print([f for f in os.listdir('/content') if f.endswith('.tsv')])

['train.tsv', 'test.tsv', 'dev.tsv']


In [20]:
TEXT_NATIVE = 'OriginalText'
LANG = 'Language'
MAXLEN = 128
import torch
print('ready')

ready


## 5b. AfriHate — inference on your corpus (OriginalText)


In [21]:
AFRIHATE_POSITIVE = {'hate'}   # which AfriHate labels count as Hate (add 'abusive' to include it)
hate_idx = {i for i,n in enumerate(LABEL_NAMES) if n.lower() in AFRIHATE_POSITIVE}
model.eval()
labels, preds = [], []
for i,t in enumerate(df[TEXT_NATIVE].tolist()):
    enc = tok(str(t)[:2000], truncation=True, max_length=MAXLEN, return_tensors='pt').to(model.device)
    with torch.no_grad():
        idx = int(model(**enc).logits.argmax(-1))
    labels.append(LABEL_NAMES[idx]); preds.append(int(idx in hate_idx))
    if (i+1)%100==0: print(f'  AfriHate {i+1}/{len(df)}')
df['afrihate_label']=labels; df['afrihate_pred']=preds
print('AfriHate Hate predictions:', sum(preds))


  AfriHate 100/838
  AfriHate 200/838
  AfriHate 300/838
  AfriHate 400/838
  AfriHate 500/838
  AfriHate 600/838
  AfriHate 700/838
  AfriHate 800/838
AfriHate Hate predictions: 280


In [24]:
df.to_csv('predictions_partial.csv', index=False)
from google.colab import files
files.download('predictions_partial.csv')
print('Saved + downloading. Pred columns:',
      [c for c in df.columns if 'pred' in c or 'score' in c or 'afrihate' in c])

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved + downloading. Pred columns: ['persp_score', 'persp_pred', 'afrihate_label', 'afrihate_pred']


In [25]:
a = df[df['afrihate_pred'].notna()].copy()
print("=== AfriHate vs your human labels (Hate vs not-Hate) ===")
print("Overall agreement:", f"{(a['afrihate_pred']==a['gold_bin']).mean():.1%}")
print("\nBy language:")
for lang in a[LANG].unique():
    s = a[a[LANG]==lang]
    print(f"  {lang}: {(s['afrihate_pred']==s['gold_bin']).mean():.1%} agreement ({len(s)} rows)")
print("\nRecall — on rows YOU labeled HATE, AfriHate also said Hate:",
      f"{(a[a.gold_bin==1]['afrihate_pred']==1).mean():.1%}")
print("False alarms — on NOT-hate rows, AfriHate said Hate:",
      f"{(a[a.gold_bin==0]['afrihate_pred']==1).mean():.1%}")

print("\n=== CAUGHT (you=Hate, AfriHate=Hate) ===")
for _,r in a[(a.gold_bin==1)&(a.afrihate_pred==1)].head(3).iterrows():
    print(" •", str(r['EnglishTranslation'])[:120])
print("\n=== MISSED (you=Hate, AfriHate=not-Hate) ===")
for _,r in a[(a.gold_bin==1)&(a.afrihate_pred==0)].head(3).iterrows():
    print(" •", str(r['EnglishTranslation'])[:120])

=== AfriHate vs your human labels (Hate vs not-Hate) ===
Overall agreement: 56.7%

By language:
  Afan Oromo: 39.1% agreement (368 rows)
  Amharic: 70.4% agreement (470 rows)

Recall — on rows YOU labeled HATE, AfriHate also said Hate: 43.0%
False alarms — on NOT-hate rows, AfriHate said Hate: 9.5%

=== CAUGHT (you=Hate, AfriHate=Hate) ===
 • the Amharas are teaming up with muslims to ruin the country
 • [Faithful.] Our Fanos, may God be your arm and shield. My brothers and sisters, you have given your precious souls for t
 • the Neftegna are the source of every problem

=== MISSED (you=Hate, AfriHate=not-Hate) ===
 • Muslims and the [SLUR_OROMO] are a threat to the city
 • [Condensed, faithful.] Asks how much has really been done — beyond media theory and political analysis — to equip Fano w
 • These ones should have been set ablaze right where they are.


In [23]:
df.to_excel('predictions_partial.xlsx', index=False)
from google.colab import files
files.download('predictions_partial.xlsx')
print('Saved + downloading. Pred columns:',
      [c for c in df.columns if 'pred' in c or 'score' in c or 'afrihate' in c])

IndexError: At least one sheet must be visible

In [17]:
import pandas as pd
df = pd.read_excel('perspective_results.xlsx')
print('df:', len(df), '| persp scored:', df['persp_score'].notna().sum())

df: 838 | persp scored: 687


In [8]:
import pandas as pd
df = pd.read_excel('perspective_results.xlsx')
print('Reloaded df:', len(df), 'rows | persp scored:', df['persp_score'].notna().sum())

FileNotFoundError: [Errno 2] No such file or directory: 'perspective_results.xlsx'

## 6. Classifier 3 — Amharic mBERT (Amharic rows only)
Check the printed `id2label` and adjust `AMMBERT_POSITIVE` so the 'hate' label maps to 1.


In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

AMM_MODEL = 'amengemeda/amharic-hate-speech-detection-mBERT'
amm_tok = AutoTokenizer.from_pretrained(AMM_MODEL)
amm_model = AutoModelForSequenceClassification.from_pretrained(AMM_MODEL)
amm_model.eval()
print('id2label:', amm_model.config.id2label)   # <-- paste me this

AMM_POSITIVE = {'hate','label_1','1','hate speech'}   # adjust after seeing id2label
hate_ids = {i for i,n in amm_model.config.id2label.items() if str(n).lower() in AMM_POSITIVE}

labels, preds = [], []
for _, row in df.iterrows():
    if str(row[LANG]).strip() != 'Amharic':
        labels.append(None); preds.append(None); continue
    enc = amm_tok(str(row[TEXT_NATIVE])[:2000], truncation=True, max_length=MAXLEN, return_tensors='pt')
    with torch.no_grad():
        idx = int(amm_model(**enc).logits.argmax(-1))
    labels.append(amm_model.config.id2label[idx]); preds.append(int(idx in hate_ids))
df['ammbert_label'] = labels; df['ammbert_pred'] = preds
print('Amharic mBERT scored', df['ammbert_pred'].notna().sum(), 'Amharic rows')

ImportError: cannot import name '_init_sox' from 'torchaudio._extension.utils' (/usr/local/lib/python3.12/dist-packages/torchaudio/_extension/utils.py)

In [3]:
!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q -U transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 57.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 79.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 10.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import pandas as pd, torch
df = pd.read_csv('predictions_partial.csv')
TEXT_NATIVE='OriginalText'; LANG='Language'; MAXLEN=128
print('Reloaded:', len(df), 'rows | cols:', [c for c in df.columns if 'pred' in c or 'score' in c])

Reloaded: 838 rows | cols: ['persp_score', 'persp_pred', 'afrihate_pred']


In [5]:
!pip install -q torch torchvision transformers --upgrade --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 246.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516

In [3]:
TEXT_NATIVE='OriginalText'; LANG='Language'; MAXLEN=128
print('ready')

ready


In [27]:
!pip install -q -U transformers

Exception ignored in: <function ZipFile.__del__ at 0x7a1c1b08a700>
Traceback (most recent call last):
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1966, in __del__
    self.close()
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1983, in close
    self.fp.seek(self.start_dir)
ValueError: seek of closed file
Exception ignored in: <function ZipFile.__del__ at 0x7a1c1b08a700>
Traceback (most recent call last):
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1966, in __del__
    self.close()
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1983, in close
    self.fp.seek(self.start_dir)
ValueError: seek of closed file


In [2]:
import pandas as pd, torch
df = pd.read_csv('predictions_partial.csv')
print('Reloaded:', len(df), 'rows | cols:', [c for c in df.columns if 'pred' in c or 'score' in c])

Reloaded: 838 rows | cols: ['persp_score', 'persp_pred', 'afrihate_pred']


## 7. Save predictions for Stage 2


In [ ]:
keep=['PostID',LANG,GOLD,'gold_bin','persp_score','persp_pred',
      'afrihate_label','afrihate_pred','ammbert_label','ammbert_pred']
out=df[[c for c in keep if c in df.columns]].copy()
out.to_excel('predictions.xlsx', index=False)
print('Saved predictions.xlsx with', len(out), 'rows.')
from google.colab import files; files.download('predictions.xlsx')


In [1]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix

df = pd.read_csv('predictions_partial.csv')
LANG = 'Language'
CLASSIFIERS = [('Perspective','persp_pred'), ('AfriHate','afrihate_pred')]

def metrics_row(name, scope, frame, pred_col):
    sub = frame[frame[pred_col].notna()]
    if len(sub) == 0: return None
    yt = sub['gold_bin'].astype(int); yp = sub[pred_col].astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0,1]).ravel()
    return {'classifier':name, 'scope':scope,
            'coverage':f"{len(sub)}/{len(frame)} ({len(sub)/len(frame):.0%})",
            'accuracy':round(accuracy_score(yt,yp),3),
            'precision':round(precision_score(yt,yp,zero_division=0),3),
            'recall':round(recall_score(yt,yp,zero_division=0),3),
            'f1':round(f1_score(yt,yp,zero_division=0),3),
            'TP':tp,'FP':fp,'FN':fn,'TN':tn}

rows = []
for name, col in CLASSIFIERS:
    rows.append(metrics_row(name,'Overall',df,col))
    for lang in ['Amharic','Afan Oromo']:
        r = metrics_row(name, lang, df[df[LANG]==lang], col)
        if r: rows.append(r)
results = pd.DataFrame([r for r in rows if r])

print("="*72)
print("STAGE 2 — CLASSIFIER EVALUATION  (positive class = Hate)")
print("="*72)
print(results.to_string(index=False))

print("\n--- Perspective coverage by language ---")
for lang in ['Amharic','Afan Oromo']:
    s = df[df[LANG]==lang]; sc = s['persp_pred'].notna().sum()
    print(f"  {lang}: {sc}/{len(s)} scored ({sc/len(s):.0%})")

results.to_csv('stage2_results.csv', index=False)
from google.colab import files
files.download('stage2_results.csv')
print("\nSaved + downloading stage2_results.csv")

STAGE 2 — CLASSIFIER EVALUATION  (positive class = Hate)
 classifier      scope       coverage  accuracy  precision  recall    f1  TP  FP  FN  TN
Perspective    Overall  687/838 (82%)     0.345      1.000   0.104 0.188  52   0 450 185
Perspective    Amharic  409/470 (87%)     0.347      1.000   0.113 0.203  34   0 267 108
Perspective Afan Oromo  278/368 (76%)     0.342      1.000   0.090 0.164  18   0 183  77
   AfriHate    Overall 838/838 (100%)     0.567      0.918   0.430 0.586 257  23 340 218
   AfriHate    Amharic 470/470 (100%)     0.704      0.914   0.640 0.753 212  20 119 119
   AfriHate Afan Oromo 368/368 (100%)     0.391      0.938   0.169 0.287  45   3 221  99

--- Perspective coverage by language ---
  Amharic: 409/470 scored (87%)
  Afan Oromo: 278/368 scored (76%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Saved + downloading stage2_results.csv
